# 05 · Evaluation & testing with synthetic data
Measures retrieval, answer correctness, grounding, citations, fail-closed behaviour and cross-subject leakage over the synthetic corpus. Also runs the pytest suite.

In [1]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env  (set NB_OFFLINE=0 or edit below)
OFFLINE = os.getenv("NB_OFFLINE", "1") == "1"
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

mode            : OFFLINE (mock)
vector store    : local (/home/claude/proj/v1/rag_service/chroma_notebook)
embedding       : hash512
LLM providers   : ['gemini', 'groq', 'openai']


In [2]:
import synthetic, pandas as pd
from app.agent import ask, NO_ANSWER
from app.store import get_store
assert get_store().count() > 0, "Run notebook 01 first"
rows = []
for subj, q, kw, src in synthetic.QA:
    o = ask(q, subject_codes=[subj])
    rows.append(dict(subject=subj, question=q, correct=kw in o["answer"].lower(), grounded=o["grounded"],
                     cited=bool(o["sources"]), conf=o["confidence"], ms=o["latency_ms"]))
ans = pd.DataFrame(rows); ans

,subject,question,correct,grounded,cited,conf,ms
0,BIO101,Where does photosynthesis take place in a plan...,True,True,True,0.982,16
1,BIO101,Which organelle produces ATP?,True,True,True,1.000,5
2,BIO101,What does semi-conservative DNA replication mean?,True,True,True,0.917,4
3,BIO101,Which enzyme adds nucleotides during DNA repli...,True,True,True,0.917,4
4,PHY101,State Newton's second law.,True,True,True,0.917,5
5,PHY101,What is the formula for kinetic energy?,True,True,True,0.894,4
6,PHY101,When does total internal reflection occur?,True,True,True,1.000,4
7,CS201,What does ACID stand for in databases?,True,True,True,1.000,5
8,CS201,What is third normal form?,True,True,True,1.000,5
9,CS201,How are deadlocks resolved?,True,True,True,0.990,4


In [3]:
print("answer correctness :", ans.correct.mean())
print("grounded            :", ans.grounded.mean())
print("has citations       :", ans.cited.mean())
print("mean confidence     :", round(ans.conf.mean(), 3), "| mean latency ms:", int(ans.ms.mean()))

answer correctness : 1.0
grounded            : 1.0
has citations       : 1.0
mean confidence     : 0.962 | mean latency ms: 5


## Fail-closed on unanswerable questions

In [4]:
un = [ask(q, subject_codes=[s]) for s, q in synthetic.UNANSWERABLE]
print("refused correctly:", sum(o["answer"] == NO_ANSWER for o in un), "/", len(un))

refused correctly: 3 / 3


## Cross-subject leakage probes

In [5]:
leaks = [ask(q, subject_codes=[s]) for s, q, _ in synthetic.LEAKAGE]
print("no leakage:", all(o["answer"] == NO_ANSWER and not o["sources"] for o in leaks))

no leakage: True


## Run the automated test-suite

In [6]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-600:])

....................................................                   [100%]
54 passed in 2.51s



### Reading the results
In **OFFLINE** mode the LLM is a rule-based mock, so these numbers prove the *plumbing, security and fail-closed logic*. Set `NB_OFFLINE=0` (real providers) and re-run to measure real answer quality; correctness below ~0.9 on this easy set usually means a chunking, embedding-model or prompt issue worth investigating.